# FoodGuard AI — Correlation Engine (LangGraph)

**Purpose**: Orchestrate the CorrelationAgent using LangGraph.

This agent:
1. Reads ML predictions (aroma/taste/vision) from DB for a batch
2. Normalizes predictions to signal tokens
3. Queries correlation_rules table for matches
4. Performs weighted voting (40% aroma + 40% taste + 20% vision)
5. Returns suspected adulterant + confidence + matched rules
6. Logs execution to agent_execution table

---

**Run this AFTER notebooks 00 & 01 (setup + data generation).**

**Can run standalone** for testing or be called by other notebooks (08_gradio_demo).

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
from typing import Dict, Any, List, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

from foodguard_lib import (
    DB_PATH,
    get_aroma_analysis,
    get_taste_analysis,
    get_vision_analysis,
    execute_query,
    insert_agent_execution,
    generate_correlation_id,
    generate_investigation_id,
    insert_investigation,
    dict_from_row
)

print("[OK] Imports successful")

## 2. Define Agent State

In [ ]:
class CorrelationAgentState(TypedDict):
    """State dictionary for CorrelationAgent."""
    # Inputs
    batch_id: str
    aroma_pred: Dict[str, Any]  # {"class": str, "confidence": float}
    taste_pred: Dict[str, Any]
    vision_pred: Dict[str, Any]

    # Intermediate states
    aroma_signal: str  # "high", "medium", "low"
    taste_signal: str
    vision_signal: str

    # Outputs
    matched_rules: List[Dict]  # List of matched rule objects
    candidate_adulterants: List[Dict]  # [{"adulterant": str, "score": float, "rules": []}]
    selected_adulterant: str
    final_confidence: float

    # Tracking
    investigation_id: Optional[str]
    execution_id: Optional[str]
    execution_time_ms: float

print("[OK] CorrelationAgentState defined")

## 3. Define CorrelationAgent Class

In [ ]:
class CorrelationAgent:
    """
    LangGraph-based correlation agent.
    Reads predictions → matches rules → weighted voting → verdict.
    """

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = db_path
        self.graph = StateGraph(CorrelationAgentState)
        self._build_graph()
        self.compiled_graph = self.graph.compile()

    def _build_graph(self):
        """Build the LangGraph workflow."""
        # Add nodes
        self.graph.add_node("normalize_signals", self.normalize_signals)
        self.graph.add_node("load_rules", self.load_rules)
        self.graph.add_node("match_rules", self.match_rules)
        self.graph.add_node("weighted_vote", self.weighted_vote)
        self.graph.add_node("finalize", self.finalize)

        # Add edges
        self.graph.add_edge("normalize_signals", "load_rules")
        self.graph.add_edge("load_rules", "match_rules")
        self.graph.add_edge("match_rules", "weighted_vote")
        self.graph.add_edge("weighted_vote", "finalize")
        self.graph.add_edge("finalize", END)

        # Set entry point
        self.graph.set_entry_point("normalize_signals")

    def normalize_signals(self, state: CorrelationAgentState) -> CorrelationAgentState:
        """
        Convert prediction confidence scores to signal tokens.
        high > 0.75, medium 0.5-0.75, low < 0.5
        """
        def confidence_to_signal(confidence: float) -> str:
            if confidence >= 0.75:
                return "high"
            elif confidence >= 0.5:
                return "medium"
            else:
                return "low"

        aroma_signal = confidence_to_signal(state["aroma_pred"]["confidence"])
        taste_signal = confidence_to_signal(state["taste_pred"]["confidence"])
        vision_signal = confidence_to_signal(state["vision_pred"]["confidence"])

        return {
            **state,
            "aroma_signal": aroma_signal,
            "taste_signal": taste_signal,
            "vision_signal": vision_signal
        }

    def load_rules(self, state: CorrelationAgentState) -> CorrelationAgentState:
        """
        Load all correlation rules from DB.
        (Could optimize to only load relevant rules, but full load is fine for demo)
        """
        query = "SELECT * FROM correlation_rules ORDER BY weight DESC"
        rows = execute_query(self.db_path, query)
        rules = [dict_from_row(row) for row in rows]

        # Parse pattern_json for each rule
        for rule in rules:
            if isinstance(rule.get("pattern_json"), str):
                rule["pattern_json"] = json.loads(rule["pattern_json"])

        state["_all_rules"] = rules  # Store in state for next node
        return state

    def match_rules(self, state: CorrelationAgentState) -> CorrelationAgentState:
        """
        Find rules that match the current signal pattern.

        Matching strategy:
        1. Exact match on all 3 signals if available
        2. Partial match on 2 signals
        3. Partial match on 1 signal
        """
        rules = state.get("_all_rules", [])
        aroma_class = state["aroma_pred"]["class"]
        taste_class = state["taste_pred"]["class"]
        vision_class = state["vision_pred"]["class"]

        matched = []
        for rule in rules:
            pattern = rule.get("pattern_json", {})

            # Check if this rule's pattern matches current predictions
            # Simple matching: if rule targets this adulterant and signals align

            # Count matches
            matches = 0
            if "aroma" in pattern and aroma_class in pattern["aroma"]:
                matches += 1
            if "taste" in pattern and taste_class in pattern["taste"]:
                matches += 1
            if "vision" in pattern and vision_class in pattern["vision"]:
                matches += 1

            # Include rule if at least 1 signal matches
            if matches > 0:
                rule["match_count"] = matches
                matched.append(rule)

        # Sort by match count (3 > 2 > 1) and by weight
        matched.sort(key=lambda r: (r["match_count"], r.get("weight", 0)), reverse=True)

        return {
            **state,
            "matched_rules": matched
        }

    def weighted_vote(self, state: CorrelationAgentState) -> CorrelationAgentState:
        """
        Aggregate confidence scores using weighted voting.
        Weight: 40% aroma + 40% taste + 20% vision

        For each adulterant mentioned in matched rules,
        compute: score = sum(confidence[modality] * weight[modality] * rule_weight)
        """
        aroma_conf = state["aroma_pred"]["confidence"]
        taste_conf = state["taste_pred"]["confidence"]
        vision_conf = state["vision_pred"]["confidence"]

        # Modality weights (fixed)
        modality_weights = {"aroma": 0.4, "taste": 0.4, "vision": 0.2}

        votes = {}  # adulterant -> {"score": float, "rules": []}

        for rule in state["matched_rules"]:
            adulterant = rule.get("target_adulterant")
            if adulterant not in votes:
                votes[adulterant] = {"score": 0.0, "rules": [], "rule_ids": []}

            # Compute rule contribution
            modality_score = (
                aroma_conf * modality_weights["aroma"] +
                taste_conf * modality_weights["taste"] +
                vision_conf * modality_weights["vision"]
            )

            rule_weight = rule.get("weight", 0.5)
            contribution = modality_score * rule_weight * rule.get("match_count", 1)

            votes[adulterant]["score"] += contribution
            votes[adulterant]["rules"].append(rule.get("rule_name", "Unknown"))
            votes[adulterant]["rule_ids"].append(rule.get("id"))

        # Normalize scores (divide by total to get 0-1 range)
        total_score = sum(v["score"] for v in votes.values())
        if total_score > 0:
            for adulterant in votes:
                votes[adulterant]["score"] /= total_score

        # Sort by score
        sorted_votes = sorted(votes.items(), key=lambda x: x[1]["score"], reverse=True)

        # Build candidate adulterants list
        candidates = [
            {
                "adulterant": adulterant,
                "score": votes_info["score"],
                "rules": votes_info["rules"],
                "rule_ids": votes_info["rule_ids"]
            }
            for adulterant, votes_info in sorted_votes
        ]

        # Select best
        if candidates:
            best = candidates[0]
            selected_adulterant = best["adulterant"]
            final_confidence = best["score"]
        else:
            selected_adulterant = "Unknown"
            final_confidence = 0.0

        return {
            **state,
            "candidate_adulterants": candidates,
            "selected_adulterant": selected_adulterant,
            "final_confidence": final_confidence
        }

    def finalize(self, state: CorrelationAgentState) -> CorrelationAgentState:
        """
        Create investigation record and log execution.
        """
        # Create investigation
        investigation_id = insert_investigation(
            batch_id=state["batch_id"],
            predicted_class=state["selected_adulterant"],
            confidence=state["final_confidence"],
            db_path=self.db_path
        )

        # Log execution
        execution_id = insert_agent_execution(
            investigation_id=investigation_id,
            agent_name="CorrelationAgent",
            input_data={
                "batch_id": state["batch_id"],
                "aroma_pred": state["aroma_pred"],
                "taste_pred": state["taste_pred"],
                "vision_pred": state["vision_pred"]
            },
            output_data={
                "selected_adulterant": state["selected_adulterant"],
                "final_confidence": state["final_confidence"],
                "candidate_adulterants": state["candidate_adulterants"]
            },
            reasoning=f"Matched {len(state['matched_rules'])} rules. Weighted vote: {state['selected_adulterant']} ({state['final_confidence']:.1%})",
            execution_time_ms=state.get("execution_time_ms", 0),
            db_path=self.db_path
        )

        return {
            **state,
            "investigation_id": investigation_id,
            "execution_id": execution_id
        }

    def invoke(self, batch_id: str) -> Dict[str, Any]:
        """
        Run the correlation agent on a batch.

        1. Fetch predictions from DB
        2. Execute the LangGraph workflow
        3. Return final state
        """
        # Fetch data from DB
        aroma = get_aroma_analysis(batch_id, db_path=self.db_path)
        taste = get_taste_analysis(batch_id, db_path=self.db_path)
        vision = get_vision_analysis(batch_id, db_path=self.db_path)

        if not (aroma and taste and vision):
            raise ValueError(f"Incomplete data for batch {batch_id}")

        # Build initial state
        initial_state = CorrelationAgentState(
            batch_id=batch_id,
            aroma_pred={
                "class": aroma.get("predicted_class"),
                "confidence": aroma.get("confidence", 0.0)
            },
            taste_pred={
                "class": taste.get("predicted_class"),
                "confidence": taste.get("confidence", 0.0)
            },
            vision_pred={
                "class": vision.get("predicted_class"),
                "confidence": vision.get("confidence", 0.0)
            },
            aroma_signal="",
            taste_signal="",
            vision_signal="",
            matched_rules=[],
            candidate_adulterants=[],
            selected_adulterant="",
            final_confidence=0.0,
            investigation_id=None,
            execution_id=None,
            execution_time_ms=0.0
        )

        # Execute
        start_time = time.time()
        final_state = self.compiled_graph.invoke(initial_state)
        execution_time_ms = (time.time() - start_time) * 1000

        final_state["execution_time_ms"] = execution_time_ms

        return final_state

print("[OK] CorrelationAgent class defined")

## 4. Instantiate Agent

In [ ]:
# Create agent instance
agent = CorrelationAgent(db_path=DB_PATH)
print("[OK] CorrelationAgent instantiated")

## 5. Test: Get a Sample Batch

Find a batch from the generated synthetic data to test on.

In [ ]:
# Get a sample batch ID
query = "SELECT DISTINCT batch_id FROM aroma_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    test_batch_id = dict_from_row(rows[0])["batch_id"]
    print(f"Test batch ID: {test_batch_id}")
else:
    print("[ERROR] No batches found. Run notebook 01 first.")

## 6. Run Agent on Test Batch

In [ ]:
print(f"Running CorrelationAgent on {test_batch_id}...\n")

result = agent.invoke(test_batch_id)

print("="*60)
print("CORRELATION AGENT RESULT")
print("="*60)
print(f"\nBatch ID: {result['batch_id']}")
print(f"Investigation ID: {result['investigation_id']}")
print(f"\nSignals:")
print(f"  Aroma: {result['aroma_signal']} (conf: {result['aroma_pred']['confidence']:.2%})")
print(f"  Taste: {result['taste_signal']} (conf: {result['taste_pred']['confidence']:.2%})")
print(f"  Vision: {result['vision_signal']} (conf: {result['vision_pred']['confidence']:.2%})")
print(f"\nMatched Rules: {len(result['matched_rules'])}")
for i, rule in enumerate(result['matched_rules'][:5], 1):
    print(f"  {i}. {rule['rule_name']} (weight: {rule['weight']:.2f})")
if len(result['matched_rules']) > 5:
    print(f"  ... and {len(result['matched_rules']) - 5} more")
print(f"\nCandidates:")
for i, cand in enumerate(result['candidate_adulterants'][:3], 1):
    print(f"  {i}. {cand['adulterant']}: {cand['score']:.1%}")
print(f"\n{'='*60}")
print(f"FINAL VERDICT:")
print(f"  Suspected Adulterant: {result['selected_adulterant']}")
print(f"  Confidence: {result['final_confidence']:.1%}")
print(f"  Execution Time: {result['execution_time_ms']:.1f}ms")
print(f"{'='*60}")

## 7. Test: Run Multiple Samples

In [ ]:
# Get 10 sample batches and run agent on each
query = "SELECT DISTINCT batch_id FROM aroma_analysis LIMIT 10"
rows = execute_query(DB_PATH, query)
batch_ids = [dict_from_row(row)["batch_id"] for row in rows]

print(f"Testing agent on {len(batch_ids)} samples...\n")

results_summary = []
for i, batch_id in enumerate(batch_ids, 1):
    try:
        result = agent.invoke(batch_id)
        results_summary.append({
            "batch": batch_id,
            "investigation_id": result["investigation_id"],
            "suspected": result["selected_adulterant"],
            "confidence": result["final_confidence"],
            "exec_time_ms": result["execution_time_ms"]
        })
        print(f"{i}. {batch_id} → {result['selected_adulterant']} ({result['final_confidence']:.1%})")
    except Exception as e:
        print(f"{i}. {batch_id} → ERROR: {e}")

print(f"\n[OK] Processed {len(results_summary)} samples")

## 8. Verify Agent Execution Logging

In [ ]:
# Check agent_execution table
query = "SELECT * FROM agent_execution WHERE agent_name = 'CorrelationAgent' LIMIT 5"
rows = execute_query(DB_PATH, query)

print(f"Agent executions logged: {len(rows)}\n")
for i, row in enumerate(rows, 1):
    exec_dict = dict_from_row(row)
    print(f"{i}. ID: {exec_dict['id']}")
    print(f"   Investigation: {exec_dict['investigation_id']}")
    print(f"   Agent: {exec_dict['agent_name']}")
    print(f"   Reasoning: {exec_dict['reasoning'][:80]}...")
    print()

## 9. Export Agent for Use by Other Notebooks

In [ ]:
# The agent is now ready to be imported and used by other notebooks (e.g., 08_gradio_demo)
# To use it elsewhere:
#
# from notebook05_correlation_engine import CorrelationAgent
# agent = CorrelationAgent()
# result = agent.invoke(batch_id)

print("[OK] CorrelationAgent ready for use by other notebooks")
print("\nUsage in other notebooks:")
print("  from foodguard_lib import ...")
print("  agent = CorrelationAgent(DB_PATH)")
print("  result = agent.invoke(batch_id)")
print("  print(result['selected_adulterant'])")

## ✅ Correlation Engine Complete!

**Summary**:
- ✓ LangGraph CorrelationAgent implemented
- ✓ 5-node workflow: normalize → load_rules → match → vote → finalize
- ✓ Weighted voting (40/40/20) working
- ✓ Correlation rules matched and scored
- ✓ Agent execution logged to DB
- ✓ Tested on synthetic data

**Next Steps**:
1. Run `06_food_safety_reasoning.ipynb` to test LLM + FoodSafetyAgent
2. Run `07_passport_certificate.ipynb` for report generation
3. Run `08_gradio_demo.ipynb` for full UI demo